# Thesis figures — risk, mortality, severity, physiology, GCS

This notebook reproduces the patient-level dataframe used in
`structured_model_first24h.ipynb` (the out-of-fold baseline risk merged with the
three LLM features) and produces the publication figures.

**It uses your real data.** Run the cells top to bottom. The only thing you may
need to adjust is the file paths in the *Load* cell to match your project layout.

**Group terminology (revised per supervisor feedback, points 1 and 2).** The
three extraction outcomes are now labelled consistently across every figure:

| Internal value (`llm_present`) | Figure label           | Meaning                                      |
|--------------------------------|------------------------|----------------------------------------------|
| `True`                         | Evidence detected      | the feature is documented in the note        |
| `False`                        | No evidence detected   | the note gives no explicit evidence of it    |
| `NaN`                          | Indeterminate          | the note is too ambiguous to assign a label  |

Only labels, panel titles, per-group n-values, axis text, medians, and legends
have been revised for readability. The underlying data and every computed number
are unchanged, so all values still match Table 6.1 and the reported statistics.

Each figure cell is preceded by a *suggested LaTeX caption* you can paste into
the thesis (point 1 also asks for more explanatory captions).

Figures written to `figures/` as both PDF (for LaTeX) and PNG (for preview):
1. `risk_separation_box` — baseline risk by feature group (lactate, shock, coma)
2. `observed_mortality_bar` — observed in-hospital mortality by group
3. `shock_severity_violin` — baseline risk across four severity levels
4. `shock_lowhigh_box` — reduced low (none/mild) vs high (moderate/severe)
5. `physio_profile` — median structured-marker profile by group
6. `gcs_coma_violin` — GCS distribution across coma groups


In [32]:
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300, "font.size": 11,
    "font.family": "serif", "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True,
    "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.titlesize": 12, "legend.fontsize": 9,
})

OUTDIR = "figures"
os.makedirs(OUTDIR, exist_ok=True)

# consistent colours across every figure
C_NEG, C_UNK, C_POS = "#4C78A8", "#F0A202", "#C44E52"
GROUP_COLORS = {"Not documented": C_NEG, "Unknown": C_UNK, "Positive": C_POS}

# Display labels (revised per supervisor feedback). Internal group values are
# unchanged; only what is printed on the figures changes.
DISPLAY = {"Not documented": "No evidence\ndetected",
           "Unknown": "Indeterminate",
           "Positive": "Evidence\ndetected"}
DISPLAY_1L = {"Not documented": "No evidence detected",
              "Unknown": "Indeterminate",
              "Positive": "Evidence detected"}

def save(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(OUTDIR, f"{name}.{ext}"), bbox_inches="tight")
    plt.close(fig)
    print("saved", name)


## Load and rebuild `df_model`

This mirrors the merge in your structured-model notebook: the per-patient
out-of-fold risk (`baseline_risk_oof`) joined to the three LLM feature columns.
If you already have `df_model` saved to CSV, just load that and skip the merge.


In [33]:
# ---------------------------------------------------------------
# OPTION A (simplest): load the df_model you already built/saved.
# If you saved it, point to it here and skip OPTION B.
# ---------------------------------------------------------------
DF_MODEL_PATH = "../Outputs/model_df_final.csv"  # <-- if you saved it

try:
    df_model = pd.read_csv(DF_MODEL_PATH)
    print("Loaded saved df_model:", df_model.shape)
except FileNotFoundError:
    df_model = None
    print("No saved df_model found — use OPTION B below.")


Loaded saved df_model: (1618, 27)


In [34]:
# ---------------------------------------------------------------
# OPTION B: rebuild df_model from the structured dataset + OOF risk
#           + the three LLM extraction result CSVs.
# Run this only if OPTION A did not find a saved file.
#
# Expects, exactly as in structured_model_first24h.ipynb:
#   df_model has columns: hadm_id, survival_status, baseline_risk_oof,
#                         and the 18 structured predictors.
#   LLM result CSVs have: hadm_id, llm_present (+ llm_severity for shock).
# ---------------------------------------------------------------
if df_model is None:
    # the structured df_model with baseline_risk_oof already computed
    df_model = pd.read_csv("../Outputs/df_model_oof.csv")   # <-- your OOF output

    lactate_path = "../LLM Feature Extraction/outputs/lactate_extraction_results_2026-04-02_14-10-58.csv"
    shock_path   = "../LLM Feature Extraction/outputs/shock_extraction_results_2026-04-10_21-57-35.csv"
    coma_path    = "../LLM Feature Extraction/outputs/coma_extraction_results_2026-05-13_22-55-31.csv"

    lactate_df = pd.read_csv(lactate_path)
    shock_df   = pd.read_csv(shock_path)
    coma_df    = pd.read_csv(coma_path)

    df_model = (df_model
        .merge(lactate_df[["hadm_id", "llm_present"]], on="hadm_id", how="inner")
        .rename(columns={"llm_present": "llm_present_lactate"}))
    df_model = (df_model
        .merge(shock_df[["hadm_id", "llm_present", "llm_severity"]], on="hadm_id", how="inner")
        .rename(columns={"llm_present": "llm_present_shock", "llm_severity": "llm_severity_shock"}))
    df_model = (df_model
        .merge(coma_df[["hadm_id", "llm_present"]], on="hadm_id", how="inner")
        .rename(columns={"llm_present": "llm_present_coma"}))
    print("Rebuilt df_model:", df_model.shape)

df_model.head()

,Unnamed: 0,hadm_id,survival_status,SBP,HR,Lactate,Hemoglobin,Bicarbonate,copd,received_epinephrine,...,anchor_age,pH,SpO2,GCS_Total,baseline_risk_oof,risk_strata_3,llm_present_lactate,llm_present_shock,llm_severity_shock,llm_present_coma
0,0,26184834,1,123.730769,71.392857,1.300000,10.900000,33.000000,1,0,...,68,7.340000,98.259259,7.625000,0.361723,low,True,NaN,NaN,NaN
1,1,29842315,1,104.840000,86.920000,2.150000,11.300000,26.000000,1,0,...,89,NaN,97.480000,15.000000,0.387654,medium,True,True,severe,False
2,2,27993048,0,108.640000,80.285714,NaN,11.350000,30.000000,0,0,...,54,7.350000,96.037037,15.000000,0.262898,low,NaN,False,none,True
3,3,25154057,1,130.120000,63.111111,4.600000,8.300000,20.000000,1,0,...,78,7.350000,99.888889,7.566667,0.724034,high,True,True,moderate,True
4,4,24614671,1,100.848039,72.111111,5.508333,10.983333,16.142857,0,0,...,78,7.234286,88.205882,6.666667,0.851821,high,True,True,severe,NaN


In [35]:
# ---------------------------------------------------------------
# Map the raw llm_present columns to canonical group labels.
# In your data llm_present is True / False / NaN  (NaN == unknown).
# ---------------------------------------------------------------
def to_group(series):
    out = series.copy().astype("object")
    out = out.where(series.notna(), "Unknown")
    out = out.replace({True: "Positive", False: "Not documented",
                       "True": "Positive", "False": "Not documented",
                       1: "Positive", 0: "Not documented"})
    return out

df_model["grp_lactate"] = to_group(df_model["llm_present_lactate"])
df_model["grp_shock"]   = to_group(df_model["llm_present_shock"])
df_model["grp_coma"]    = to_group(df_model["llm_present_coma"])

ORDER = ["Not documented", "Unknown", "Positive"]
for col in ["grp_lactate", "grp_shock", "grp_coma"]:
    print(col, df_model[col].value_counts().to_dict())


grp_lactate {'Positive': 790, 'Unknown': 598, 'Not documented': 230}
grp_shock {'Positive': 1226, 'Not documented': 222, 'Unknown': 170}
grp_coma {'Not documented': 756, 'Positive': 615, 'Unknown': 247}


## Figure 1 — Risk separation box plots (Fig. 6.2)

*Suggested LaTeX caption:*

> Baseline out-of-fold mortality risk by extracted feature group, shown
> separately for lactate, shock, and coma. Each box summarises the per-patient
> baseline risk from the structured first-24-hour logistic-regression model: the
> line is the group median (printed beside each box), the box spans the
> interquartile range, and whiskers extend to 1.5x IQR (outliers hidden). The
> three groups are *evidence detected*, *no evidence detected*, and
> *indeterminate*, with per-group patient counts (n) on the axis. For all three
> features the evidence-detected group sits at higher baseline risk than the
> no-evidence-detected group; the indeterminate group behaves differently across
> features and is notably high for coma.

In [36]:
RISK = "baseline_risk_oof"
feat_cols = {"Lactate": "grp_lactate", "Shock": "grp_shock", "Coma": "grp_coma"}

fig, axes = plt.subplots(1, 3, figsize=(11.5, 4.2), sharey=True)
for ax, (feat, col) in zip(axes, feat_cols.items()):
    data = [df_model.loc[df_model[col] == g, RISK].dropna().values for g in ORDER]
    n    = [int((df_model[col] == g).sum()) for g in ORDER]
    bp = ax.boxplot(data, patch_artist=True, widths=0.6, showfliers=False,
                    medianprops=dict(color="black", lw=1.6))
    for patch, g in zip(bp["boxes"], ORDER):
        patch.set_facecolor(GROUP_COLORS[g]); patch.set_alpha(0.75)
    for i, d in enumerate(data, start=1):          # median value beside each box
        if len(d):
            ax.text(i + 0.34, np.median(d), f"{np.median(d):.2f}",
                    va="center", ha="left", fontsize=8)
    ax.set_title(feat, fontsize=12, fontweight="bold")
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels([f"{DISPLAY[g]}\n(n={ni})" for g, ni in zip(ORDER, n)],
                       fontsize=8)
    ax.set_ylim(0, 1)
axes[0].set_ylabel("Baseline mortality risk\n(out-of-fold estimate)")
save(fig, "risk_separation_box")


saved risk_separation_box


## Figure 2 — Observed mortality bars (Fig. 6.3)

*Suggested LaTeX caption:*

> Observed in-hospital mortality by extracted feature group. Bars show the
> proportion of each group that died in hospital (1.0 = the whole group died);
> the mortality value and group size (n) are annotated on each bar. Across
> features the evidence-detected groups show higher observed mortality than the
> no-evidence-detected groups. For coma, the indeterminate group shows the
> highest observed mortality of any group in the study.

In [37]:
# survival_status: 1 = died in hospital (confirmed in the modelling notebook,
# where the confusion-matrix classes are 0 = survived, 1 = died). So the group
# mean of survival_status is the in-hospital mortality rate.
OUTCOME = "survival_status"
fig, ax = plt.subplots(figsize=(8.5, 4.8))
feats = list(feat_cols.keys()); x = np.arange(len(feats)); w = 0.26
for i, g in enumerate(ORDER):
    vals, ns = [], []
    for feat in feats:
        col = feat_cols[feat]
        sub = df_model[df_model[col] == g]
        vals.append(sub[OUTCOME].mean() if len(sub) else np.nan)
        ns.append(len(sub))
    bars = ax.bar(x + (i - 1) * w, vals, w, label=DISPLAY_1L[g],
                  color=GROUP_COLORS[g], alpha=0.85)
    labels = [f"{v:.2f}\n(n={ni})" for v, ni in zip(vals, ns)]
    ax.bar_label(bars, labels=labels, padding=2, fontsize=7.5)
ax.set_xticks(x); ax.set_xticklabels(feats, fontsize=11)
ax.set_ylabel("Observed in-hospital mortality\n(proportion of group who died)")
ax.set_ylim(0, 1)
ax.legend(frameon=False, ncol=3, loc="lower center",
          bbox_to_anchor=(0.5, 1.005))
save(fig, "observed_mortality_bar")


saved observed_mortality_bar


## Figure 3 — Shock severity (four-level) baseline risk (Fig. 6.4)

*Suggested LaTeX caption:*

> Baseline out-of-fold mortality risk across the four shock-severity levels
> returned by the prompt (none, mild, moderate, severe), with per-level patient
> counts (n). The median of each level is printed beside the violin. The grey
> brackets mark the reduced *low* (none/mild) versus *high* (moderate/severe)
> grouping used in the following analysis: none and mild overlap closely, as do
> moderate and severe, which motivates the two-level reduction. The
> Kruskal-Wallis and Spearman-trend statistics are reported in the text.

In [38]:
levels = ["none", "mild", "moderate", "severe"]
sev = df_model["llm_severity_shock"].astype("object").str.lower()
data = [df_model.loc[sev == lv, RISK].dropna().values for lv in levels]
n    = [len(d) for d in data]

fig, ax = plt.subplots(figsize=(7.5, 4.6))
parts = ax.violinplot(data, showmedians=True, widths=0.6)
sev_colors = ["#9ecae1", "#6baed6", "#fb6a4a", "#cb181d"]
for body, cc in zip(parts["bodies"], sev_colors):
    body.set_facecolor(cc); body.set_alpha(0.7); body.set_edgecolor("black")
for k in ("cmedians", "cbars", "cmins", "cmaxes"):
    if k in parts: parts[k].set_color("black"); parts[k].set_linewidth(1.0)
_bbox = dict(boxstyle="round,pad=0.15", facecolor="white", edgecolor="none", alpha=0.85)
for i, d in enumerate(data, start=1):        # median label sits just past the violin edge
    if len(d):
        ax.text(i + 0.38, np.median(d), f"{np.median(d):.2f}",
                va="center", ha="left", fontsize=8, bbox=_bbox)
ax.set_xlim(0.4, 4.7)
ax.set_xticks([1, 2, 3, 4])
ax.set_xticklabels([f"{lv.capitalize()}\n(n={ni})" for lv, ni in zip(levels, n)])
ax.set_ylabel("Baseline mortality risk\n(out-of-fold estimate)"); ax.set_ylim(0, 1.15)
yb = 1.03                                          # brackets sit above the violins
ax.annotate("", xy=(1, yb), xytext=(2, yb), arrowprops=dict(arrowstyle="-", color="gray"))
ax.annotate("", xy=(3, yb), xytext=(4, yb), arrowprops=dict(arrowstyle="-", color="gray"))
ax.text(1.5, yb + 0.02, "low severity", ha="center", fontsize=9, color="gray")
ax.text(3.5, yb + 0.02, "high severity", ha="center", fontsize=9, color="gray")
save(fig, "shock_severity_violin")

# statistics (console only; data unchanged)
from scipy.stats import kruskal, spearmanr
H, pH = kruskal(*[d for d in data if len(d)])
sev_num = sev.map({"none": 0, "mild": 1, "moderate": 2, "severe": 3})
mask = sev_num.notna() & df_model[RISK].notna()
rho, pr = spearmanr(sev_num[mask].astype(float), df_model.loc[mask, RISK])
print(f"Kruskal-Wallis H={H:.2f}, p={pH:.3e}")
print(f"Spearman rho={rho:.3f}, p={pr:.3e}")


saved shock_severity_violin
Kruskal-Wallis H=144.94, p=3.246e-31
Spearman rho=0.298, p=7.592e-31


## Figure 4 — Reduced low-vs-high shock severity (Fig. 6.5)

*Suggested LaTeX caption:*

> Baseline out-of-fold mortality risk under the reduced *low* (none/mild) versus
> *high* (moderate/severe) shock-severity encoding, with group sizes (n) and
> medians annotated. The two-sided Mann-Whitney comparison is shown above the
> boxes. The reduced encoding retains essentially all of the risk separation while
> removing the unstable none/mild boundary.

In [39]:
low  = df_model.loc[sev.isin(["none", "mild"]), RISK].dropna().values
high = df_model.loc[sev.isin(["moderate", "severe"]), RISK].dropna().values

from scipy.stats import mannwhitneyu
u, p = mannwhitneyu(low, high, alternative="two-sided")

fig, ax = plt.subplots(figsize=(5.8, 4.6))
bp = ax.boxplot([low, high], patch_artist=True, widths=0.55, showfliers=False,
                medianprops=dict(color="black", lw=1.6))
for patch, cc in zip(bp["boxes"], ["#6baed6", "#cb181d"]):
    patch.set_facecolor(cc); patch.set_alpha(0.75)
for i, d in enumerate([low, high], start=1):
    ax.text(i + 0.32, np.median(d), f"{np.median(d):.2f}",
            va="center", ha="left", fontsize=9)
ax.set_xticks([1, 2])
ax.set_xticklabels([f"Low severity\n(none / mild)\nn={len(low)}",
                    f"High severity\n(moderate / severe)\nn={len(high)}"])
ax.set_ylabel("Baseline mortality risk\n(out-of-fold estimate)"); ax.set_ylim(0, 1)
y = 0.90                                            # significance bracket
ax.plot([1, 1, 2, 2], [y, y + 0.02, y + 0.02, y], lw=1.0, color="black")
p_txt = "p < 0.001" if p < 0.001 else f"p = {p:.3g}"
ax.text(1.5, y + 0.03, f"Mann-Whitney, {p_txt}", ha="center", fontsize=9)
save(fig, "shock_lowhigh_box")

print(f"low vs high Mann-Whitney U={u:.1f}, p={p:.3e}")
print(f"median low={np.median(low):.4f}  median high={np.median(high):.4f}")


saved shock_lowhigh_box
low vs high Mann-Whitney U=136684.0, p=4.475e-33
median low=0.3751  median high=0.5652


## Figure 5 — Physiological profile by group (Fig. 6.6)

*Suggested LaTeX caption:*

> Median structured-marker profile across the *no evidence detected*,
> *indeterminate*, and *evidence detected* groups for each feature, with
> per-group patient counts (n). From left to right: structured lactate for the
> lactate grouping; systolic blood pressure and structured lactate for the shock
> grouping; and GCS total for the coma grouping. The position of the
> indeterminate group relative to the other two differs by feature: intermediate
> for lactate, close to no-evidence-detected for shock, and close to
> evidence-detected for coma.

In [40]:
# (marker column, grouping column, panel title, y-axis label with unit)
PHYSIO = [
    ("Lactate",   "grp_lactate", "Lactate group",  "Structured lactate (mmol/L)"),
    ("SBP",       "grp_shock",   "Shock group \u00b7 SBP",     "SBP (mmHg)"),
    ("Lactate",   "grp_shock",   "Shock group \u00b7 lactate", "Structured lactate (mmol/L)"),
    ("GCS_Total", "grp_coma",    "Coma group",      "GCS total"),
]
fig, axes = plt.subplots(1, len(PHYSIO), figsize=(14, 4.4))
for ax, (marker, gcol, title, ylab) in zip(axes, PHYSIO):
    vals = [df_model.loc[df_model[gcol] == g, marker].median() for g in ORDER]
    ns   = [int((df_model[gcol] == g).sum()) for g in ORDER]
    bars = ax.bar(range(3), vals, color=[GROUP_COLORS[g] for g in ORDER], alpha=0.85)
    ax.bar_label(bars, fmt="%.2f", fontsize=8, padding=2)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_ylabel(ylab, fontsize=10)
    ax.set_xticks(range(3))
    ax.set_xticklabels([f"{DISPLAY[g]}\n(n={ni})" for g, ni in zip(ORDER, ns)],
                       fontsize=7.5)
fig.tight_layout()
save(fig, "physio_profile")


saved physio_profile


## Figure 6 — GCS distribution across coma groups (Fig. 6.7)

*Suggested LaTeX caption:*

> Distribution of GCS total across the coma extraction groups, with per-group
> patient counts (n) and medians annotated. Lower GCS indicates worse
> neurological status. The indeterminate group lies between the
> no-evidence-detected and evidence-detected groups but clearly within the
> impaired range, consistent with a distinct high-risk subgroup rather than
> uninformative missing data.

In [41]:
fig, ax = plt.subplots(figsize=(7, 4.6))
data = [df_model.loc[df_model["grp_coma"] == g, "GCS_Total"].dropna().values for g in ORDER]
n    = [int((df_model["grp_coma"] == g).sum()) for g in ORDER]
parts = ax.violinplot(data, showmedians=True, widths=0.6)
for body, g in zip(parts["bodies"], ORDER):
    body.set_facecolor(GROUP_COLORS[g]); body.set_alpha(0.75); body.set_edgecolor("black")
for k in ("cmedians", "cbars", "cmins", "cmaxes"):
    if k in parts: parts[k].set_color("black"); parts[k].set_linewidth(1.0)
_bbox = dict(boxstyle="round,pad=0.15", facecolor="white", edgecolor="none", alpha=0.85)
for i, d in enumerate(data, start=1):        # median label sits just past the violin edge
    if len(d):
        ax.text(i + 0.38, np.median(d), f"{np.median(d):.1f}",
                va="center", ha="left", fontsize=9, bbox=_bbox)
ax.set_xlim(0.4, 3.7)
ax.set_xticks([1, 2, 3])
ax.set_xticklabels([f"{DISPLAY[g]}\n(n={ni})" for g, ni in zip(ORDER, n)])
ax.set_ylabel("GCS total  (3 = worst, 15 = best)"); ax.set_ylim(2, 16)
ax.text(0.02, 0.02, "lower = worse neurological status", transform=ax.transAxes,
        fontsize=8, color="gray", style="italic")
save(fig, "gcs_coma_violin")


saved gcs_coma_violin
